# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Binary classification.**

My lane is predicting which blog posts will rank — specifically, whether a post reaches **page 1** (top ~10) of search results or not. I considered framing this as scoring instead (a continuous "ranking potential" number, similar to the dataset's existing `health_score`), but classification is the more defensible choice here: exact search position is noisy and heavy-tailed (the gap between position 14 and 18 isn't meaningfully different in practice), while the page-1 threshold is the line that actually matters for whether a post gets found at all. This keeps the target well-defined and the evaluation straightforward.

In [3]:
# Section 1 has no computation of its own — the task-type decision is argued above.
# The evidence for *why* classification (not a fixed rule, not raw regression) is in Sections 4-5 below.
print("Task type: binary classification (is_page_1: yes/no)")

Task type: binary classification (is_page_1: yes/no)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_page_1`**

I derive the target directly from `position_tier`: `is_page_1 = 1` when `position_tier == "page_1"`, else `0`. This is an **observed outcome**, not an invented proxy — `avg_position` and `position_tier` come from real (anonymized) Google Search Console data, so the label reflects what actually happened rather than a rule I made up.

One honest caveat: this is a snapshot of *current* ranking state, not a forecast of ranking after future changes to a post. So predictions from this model should be read as "associated with page-1 status," not "will cause a post to rank" — a directional, decision-support signal, not a guarantee.

In [4]:
import pandas as pd
import os

candidates = [
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
    "content_refresh_anonymized.csv",
]
path = next((p for p in candidates if os.path.exists(p)), None)
if path is None:
    raise FileNotFoundError(
        "Could not find content_refresh_anonymized.csv. "
        "Update the `candidates` list above with the correct path in your repo (likely data/raw/)."
    )

df = pd.read_csv(path)
df["is_page_1"] = (df["position_tier"] == "page_1").astype(int)

print(df["is_page_1"].value_counts(normalize=True).rename("share"))

is_page_1
0    0.6062
1    0.3938
Name: share, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success metric: ROC-AUC (primary), F1 (secondary)**

Classes are moderately balanced (roughly 39% page-1, 61% not — measured below), so plain accuracy is a weak metric: a model could look decent while barely beating the majority-class baseline. ROC-AUC measures how well the model ranks page-1 posts above non-page-1 posts across all thresholds, which fits a realistic use case — "rank all posts by likelihood of reaching page 1, then review the top of that list" — without forcing a single arbitrary cutoff up front. F1 is a useful secondary check once a threshold is chosen for a concrete yes/no decision (e.g. "flag this post for optimization").

In [5]:
majority_baseline = max(df["is_page_1"].mean(), 1 - df["is_page_1"].mean())
print(f"Majority-class baseline accuracy: {majority_baseline:.3f}")
print("Any model needs to clear this baseline; ROC-AUC and F1 will be the numbers reported against it.")

Majority-class baseline accuracy: 0.606
Any model needs to clear this baseline; ROC-AUC and F1 will be the numbers reported against it.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis: one row = one blog post (content page).**

Each row in `content_refresh_anonymized.csv` is a single published content page, identified by `content_id`, with pre-refresh content attributes (word count, age, intent, freshness) alongside measured ranking/engagement outcomes (avg_position, ctr, impressions). The dataframe below confirms the shape and shows a representative slice of columns, including the derived target.

In [6]:
print("Shape:", df.shape)

preview_cols = [
    "content_id", "word_count", "content_age_days", "search_volume",
    "competition_level", "avg_position", "position_tier", "is_page_1",
]
df[preview_cols].head(10)

Shape: (30000, 45)


,content_id,word_count,content_age_days,search_volume,competition_level,avg_position,position_tier,is_page_1
0,content_304f48230142,3221.0,187,10.0,HIGH,10.6,striking,0
1,content_a1fb4e703a9e,2481.0,445,90.0,LOW,20.3,page_3_5,0
2,content_9aa793d4d895,3515.0,141,0.0,LOW,36.5,page_3_5,0
3,content_331d6c4de07b,NaN,463,10.0,LOW,6.2,page_1,1
4,content_d99b7a2d90ca,2803.0,263,0.0,LOW,44.0,page_3_5,0
5,content_d4084a4bc775,3080.0,147,720.0,HIGH,8.5,page_1,1
6,content_9a34b442b552,3059.0,90,0.0,LOW,7.0,page_1,1
7,content_a63219c6e95a,NaN,445,590.0,MEDIUM,21.2,page_3_5,0
8,content_5e6c160719bc,3807.0,90,0.0,LOW,46.0,page_3_5,0
9,content_c27558df2b0c,NaN,257,0.0,LOW,4.9,page_1,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**A fixed rule performs worse than a coin flip on the majority class.**

I tested the most obvious fixed rule an SEO team might reach for — "longer content ranks better, so predict page-1 for above-median word count." It scored only ~45.7% accuracy, *worse* than simply always predicting the majority class (~60.6%, measured in Section 3).

The correlation table below shows why: no single feature (word count, content age, days since update, search volume, cpc) has more than a weak, and sometimes counter-intuitive, relationship with page-1 status — word count is actually slightly *negatively* correlated with reaching page 1. Ranking is a directional outcome shaped by several weak, interacting signals at once, not one dominant variable with a clean threshold. That's exactly the pattern a fixed if-statement can't capture but a model that learns from combinations of features can.

In [7]:
import numpy as np

# Naive fixed rule: "longer content ranks better"
median_wc = df["word_count"].median()
df["rule_pred"] = (df["word_count"] > median_wc).astype(int)
valid = df.dropna(subset=["word_count"])
rule_acc = (valid["rule_pred"] == valid["is_page_1"]).mean()
print(f"Naive rule (word_count > median) accuracy: {rule_acc:.3f}")
print(f"Majority-class baseline accuracy:          {majority_baseline:.3f}")
print()

num_cols = ["word_count", "char_count", "content_age_days",
            "days_since_last_update", "search_volume", "cpc"]
print("Correlation of individual features with is_page_1:")
print(df[num_cols + ["is_page_1"]].corr()["is_page_1"].drop("is_page_1").sort_values())

Naive rule (word_count > median) accuracy: 0.457
Majority-class baseline accuracy:          0.606

Correlation of individual features with is_page_1:
word_count               -0.145243
char_count               -0.125125
content_age_days         -0.081305
days_since_last_update   -0.036006
search_volume            -0.029951
cpc                      -0.029291
Name: is_page_1, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.